# Modular library for VMC with JAX

In this library we document and test the modular parts of our library for Quantum Monte Carlo. In order for the system to be scallable to many-body systems (heavy atoms or molecules) and allow the integration of RL techniques, we defined a strict modular code.

The library is separated conceptually:

1. **Wavefunctions**: Neural networks (can be from DeepQMC or defined in FLAX LINEN) (TODO: NNX)
2. **Hamiltonians**: Defines the physical syustem and computes local energies.
3. **Samplers**: The stochastic engines for exploration of the configuration spaces of the system (Metropolis-Hastings, MALA, RL-MALA)
4. **Trainers**: The weight update logics and XLA optimization.

## Samplers

All Metropolis-Hastings follow a universal accept / reject rule to guarantee that the Markov chain convergex to the goal distribution $|\Psi(x)|^2$. The acceptation probability of a jump $x\rightarrow x'$ is given by,

$$A(x \to x') = \min\left(1, \frac{|\Psi(x')|^2 q(x|x')}{|\Psi(x)|^2 q(x'|x)}\right)$$

where $q(x' | x)$ is the probability of proposing the new state $x'$ giben that the current state is $x$. Working with logs, in order to avoid numerical instability,

$$\log A = 2(\log|\Psi(x')| - \log|\Psi(x)|) + \log q(x|x') - \log q(x'|x)$$

In order to implement this in JAX, we design a class $\texttt{BaseSampler}$ that implements the foundations of all the samplers with Metropolis acceptation rule (TODO: Generalize for other sampling rules). This uses the $\texttt{jax.jit}$ function.

### Random Walk Metropolis-Hastings: $\texttt{MetropolisHastingsSampler}$

The $\texttt{samplers.MetropolisHastingsSampler}$ is the basic RWMH sampler. Here, the proposal is a isotropycal jump guided by Gaussian noise,

$$x' = x + \sigma \eta$$

Given that the probability for $x \rightarrow x'$ is the same as for $x' \rightarrow x$, the transition kernel is symmetric: $q(x'|x) = q(x|x')$. So, we have that,

$$\log\left(\frac{q(x|x')}{q(x'|x)}\right) = 0.0$$

then, we can compute the acceptation probability only with the value of the log of the wavefunction.

### Metropolis-Adjusted Langevin Algorithm: $\texttt{MALASampler}$

MALA uses the gradient of the probability distribution to guide the proposals to the high-probability regions:

$$x' = x + \tau \nabla \log|\Psi(x)|^2 + \sqrt{2\tau} \eta$$

where $\eta$ is Gaussian noise.

When we add the gradient to the proposal, the kernel becomes anti-symmetrical. The transition probability follows a Gaussian distribution centered in the drift's magnitude:

$$\log q(x'|x) = -\frac{1}{4\tau} \left\| x' - x - \tau \nabla \log|\Psi(x)|^2 \right\|^2 + C$$

The class $\texttt{MALASampler}$ extracts the gradient, proposes a new position and computes the proposed gradients, in order to compute the transition kernel. We clip the gradients in order to minimize the probability of numerical instability.

#### Example of usage

First, we need to define a probability distribution to sample from. Since we want to study physical systems, and we want to search for the ground truth (TODO: excited states), we want to work with the wavefunction of the system, an ansatz, and then minimize its energy by optimizing a set of variational parameters. 

In our library, we need the wavefunction to return the natural logarithm of the wavefunction. For example, imagine we want to sample the position distribution from a quantum system that is in the state:

$$\Psi(x) = \exp\left(-\alpha x^2\right)$$

then, in order to do it with our library, we need to define the $\ln \Psi(x)$ as follows:

In [1]:
import jax.numpy as jnp
import jax

def dummy_ansatz(params, x):
    
    # gaussian wavefunction: log(exp(-alpha * x^2)) = -alpha * x^2
    alpha = params.get("alpha", 1.0) # if the dictionary "params" does not have a key called "alpha", it is defined as 1.0

    return jnp.sum(-alpha * x**2, axis = (-1, -2))

def sum_dummy_ansatz(params, x):
    return jnp.sum(dummy_ansatz(params, x))

In [2]:
key = jax.random.PRNGKey(42)
BATCH_SIZE = 128
N_ELEC = 2
DIMS = 3

walkers_init = jax.random.normal(key, (BATCH_SIZE, N_ELEC, DIMS))
dummy_params = {"ansatz": {"alpha": 0.5}}

log_psi_init_curr = dummy_ansatz(dummy_params["ansatz"], walkers_init)
grad_init_curr = jax.grad(lambda w: sum_dummy_ansatz(dummy_params["ansatz"], w))(walkers_init)

cache_init = {
    "log_psi_curr": log_psi_init_curr,
    "grad_curr": grad_init_curr
}

In [3]:
print("Initializing JIT tests")

from samplers import MetropolisHastingsSampler, RLMALASampler, MALASampler

from wavefunctions import DeepQMCAnsatz
from system import PhysicalSystem

# Ejemplo de uso rápido:
lih_system = PhysicalSystem(
    name="LiH",
    n_elec=4,
    n_up=2,
    n_dn=2,
    coords=jnp.array([[0.0, 0.0, 0.0], [3.015, 0.0, 0.0]]),
    charges=jnp.array([3, 1], dtype=jnp.int32)
)
print(f"Sistema creado: {lih_system.name} | Espín: {lih_system.spin}")


# test 1. random walk metropolis hastings
MHSampler = MetropolisHastingsSampler(dummy_ansatz, BATCH_SIZE, sigma = 0.1, system=lih_system)
key, k_step = jax.random.split(key)
new_w_mh, stats_mh, new_cache_mh, _ = MHSampler.step(k_step, walkers_init, dummy_params, cache_init)
print(f"Metropolis Hastings Algorithm -> Acc: {stats_mh["acceptance"]:.4f} | EJD: {stats_mh["ejd"]:.4f}")

# test 2. Classic MALA
mala = MALASampler(dummy_ansatz, BATCH_SIZE, tau=0.01, system=lih_system)
key, k_step = jax.random.split(key)
new_w_mala, stats_mala, new_cache_mala, _ = mala.step(k_step, walkers_init, dummy_params, cache_init)
print(f"Metropolis - Accelerated Langevin Algorithm -> Acc: {stats_mala["acceptance"]:-4f} | EJD: {stats_mala['ejd']:.4f}")

JAX was imported before deepqmc, TensorFloat32 precision might be enabled. You may experience numerical issues.


Initializing JIT tests


/home/romanarmenta/miniconda3/envs/jax-gpu/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sistema creado: LiH | Espín: 0
Metropolis Hastings Algorithm -> Acc: 0.8750 | EJD: 0.1388
Metropolis - Accelerated Langevin Algorithm -> Acc: 0.898438 | EJD: 0.2001


In [10]:
# define an actor
def dummy_actor(params_actor, walkers, elec_idx, static_state, exploration_mode):
    """
    Emulates the cbehavior of a flax_module.apply. This function recieves the positions and 
    returns the drift and log-variance matrix
    """

    # pick an specific electron (vmapp will do the batch)
    r_i = walkers[:, elec_idx, :]

    explore_factor = jnp.where(exploration_mode, 1.0, 0.0)

    # this dummy drift pushes the electrons through the origin
    drift = -params_actor["push"] * r_i * explore_factor

    # variance matrix
    sigma_raw = jnp.zeros_like(r_i)

    return drift, sigma_raw

dummy_params["actor"] = {"push": 0.01}

print(dummy_params)
print("initialization JIT for RLMALA")

rl_mala = RLMALASampler(
    ansatz_apply = dummy_ansatz,
    actor_apply = dummy_actor,
    batch_size = BATCH_SIZE,
    tau = 0.01,
    gamma = 0.04,
    beta = 10.,
    system = lih_system # in our toy model there is no fixed info needed
)

key, k_step = jax.random.split(key)
new_w_rl, stats_rl, new_cache_rl, aux_rl = rl_mala.step(
    k_step, 
    walkers_init, 
    dummy_params, 
    cache_init, 
    exploration_mode=True
)

print(f"RL-MALA -> Acceptation: {stats_rl["acceptance"]:.4f} | EJD: {stats_rl["ejd"]:4f}")
print(f"         -> Drift MALA: {stats_rl['mag_mala']:.4f} | Drift Actor: {stats_rl['mag_actor']:.4f}")

# check if everyting is saved in stats:
rl_data = aux_rl["rl_data"]
print(f"         -> RL log_q Fwd shape: {rl_data['log_q_fwd_policy'].shape} (correct if equals [{BATCH_SIZE}])")

{'ansatz': {'alpha': 0.5}, 'actor': {'push': 0.01}}
initialization JIT for RLMALA
RL-MALA -> Acceptation: 0.0078 | EJD: 0.002583
         -> Drift MALA: 0.0156 | Drift Actor: 0.0156
         -> RL log_q Fwd shape: (128,) (correct if equals [128])


In [5]:
from wavefunctions import DeepQMCAnsatz
from system import PhysicalSystem

# Ejemplo de uso rápido:
lih_system = PhysicalSystem(
    name="LiH",
    n_elec=4,
    n_up=2,
    n_dn=2,
    coords=jnp.array([[0.0, 0.0, 0.0], [3.015, 0.0, 0.0]]),
    charges=jnp.array([3, 1], dtype=jnp.int32)
)
print(f"Sistema creado: {lih_system.name} | Espín: {lih_system.spin}")


Sistema creado: LiH | Espín: 0


In [6]:
# =======================================================
# PRUEBA DE INTEGRACIÓN
# =======================================================
print("Cargando el Ansatz de DeepQMC para LiH...")
# Usamos el lih_system creado en la celda anterior
ansatz = DeepQMCAnsatz(lih_system)

# Generamos un batch de caminantes aleatorios para probar
key = jax.random.PRNGKey(42)
test_walkers = jax.random.normal(key, (10, lih_system.n_elec, 3)) # Batch de 10

print("Inicializando pesos (Esto puede tomar unos segundos al compilar)...")
params_psi = ansatz.init(key, test_walkers)

print("Evaluando caminantes...")
log_psi_vals = ansatz.apply(params_psi, test_walkers)

print(f"Shape del output log|Psi|: {log_psi_vals.shape} (Debe ser (10,))")
print(f"Valores de prueba: {log_psi_vals[:3]}")

Cargando el Ansatz de DeepQMC para LiH...
Inicializando pesos (Esto puede tomar unos segundos al compilar)...
Evaluando caminantes...
Shape del output log|Psi|: (10,) (Debe ser (10,))
Valores de prueba: [-5.417486  -8.454476  -7.6821623]


In [11]:
from hamiltonians import FullCoulombHamiltonian

# =======================================================
# PRUEBA DE INTEGRACIÓN
# =======================================================
print(f"Probando Hamiltoniano Coulombiano para {lih_system.name}...")
hamiltonian = FullCoulombHamiltonian(lih_system)

print(f"Repulsión constante Núcleo-Núcleo (V_NN) precalculada: {hamiltonian.V_NN:.4f} Ha")

# Evaluamos la energía potencial pura en los caminantes de prueba
v_pot = hamiltonian.potential_energy(test_walkers)
print(f"Shape de Energía Potencial: {v_pot.shape} (Debe ser (10,))")
print(f"Valores V_pot de los primeros 3 caminantes: {v_pot[:3]}")

# (Opcional) Probar la Energía Local completa usando el Ansatz Dummy
# dummy_params = {"ansatz": {"alpha": 0.5}}
e_loc = hamiltonian.local_energy(dummy_ansatz, dummy_params["ansatz"], test_walkers)
print(f"Energía Local completa (Dummy): {e_loc[:3]}")

Probando Hamiltoniano Coulombiano para LiH...
Repulsión constante Núcleo-Núcleo (V_NN) precalculada: 0.9950 Ha
Shape de Energía Potencial: (10,) (Debe ser (10,))
Valores V_pot de los primeros 3 caminantes: [-15.023013  -5.031621  -5.115226]
Energía Local completa (Dummy): [-11.816632   -5.264412   -2.8894303]


In [13]:
import optax
from trainers import GroundStateTrainer

# =======================================================
# PRUEBA DE INTEGRACIÓN FINAL: EL ECOSISTEMA COMPLETO
# =======================================================
print("Iniciando Ensamblaje del Ecosistema RL-VMC...")

# 1. EL SISTEMA (Definimos un LiH de prueba)
test_system = PhysicalSystem(
    name="LiH_Test",
    n_elec=4,
    n_up=2, n_dn=2,
    coords=jnp.array([[0.0, 0.0, 0.0], [3.015, 0.0, 0.0]]),
    charges=jnp.array([3, 1], dtype=jnp.int32)
)

# 2. EL HAMILTONIANO (Física universal de Coulomb)
hamiltonian = FullCoulombHamiltonian(test_system)

# 3. LOS OPTIMIZADORES (Optax standard)
opt_ansatz = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(5e-4))
opt_actor = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(1e-3))

# Inicializamos estados ficticios de optimizador (usando nuestros pesos dummy)
opt_state_psi = opt_ansatz.init(dummy_params["ansatz"])
opt_state_actor = opt_actor.init(dummy_params["actor"])

# 4. EL SAMPLER (Híbrido)
rl_sampler = RLMALASampler(
    ansatz_apply=dummy_ansatz,
    actor_apply=dummy_actor,
    batch_size=BATCH_SIZE,
    tau=0.01,
    gamma=0.04,
    beta=10.0,
    system=test_system # Inyectamos la dependencia
)

# 5. EL ENTRENADOR
trainer = GroundStateTrainer(
    system=test_system,
    hamiltonian=hamiltonian,
    sampler=rl_sampler,
    opt_ansatz=opt_ansatz,
    opt_actor=opt_actor
)

print("\n--- Compilando y ejecutando pasos de entrenamiento ---")

# Simulamos 3 pasos de entrenamiento para comprobar la estabilidad del caché y JIT
current_walkers = walkers_init  # (Del batch que creamos en pruebas anteriores)
current_params_psi = dummy_params["ansatz"]
current_params_actor = dummy_params["actor"]
current_cache = cache_init

print("\n--- Compilando y ejecutando pasos de entrenamiento ---")

# 1. Generamos NUEVOS caminantes que respeten n_elec=4 de nuestro test_system
key = jax.random.PRNGKey(99)
key, k_walk = jax.random.split(key)
current_walkers = jax.random.normal(k_walk, (BATCH_SIZE, test_system.n_elec, 3)) 

# 2. Asignamos los parámetros dummy
current_params_psi = dummy_params["ansatz"]
current_params_actor = dummy_params["actor"]

# 3. Inicializamos el caché desde cero para estos nuevos caminantes de 4 electrones
curr_log_psi_init = dummy_ansatz(current_params_psi, current_walkers)
# Recuerda usar la función suma para el gradiente
curr_grad_init = jax.grad(lambda w: jnp.sum(dummy_ansatz(current_params_psi, w)))(current_walkers)

current_cache = {
    "curr_log_psi": curr_log_psi_init, 
    "curr_grad": curr_grad_init
}

# 4. Bucle de entrenamiento
for step in range(3):
    key, k_step1, k_step2 = jax.random.split(key, 3)
    
    # A) Paso de Optimización del Ansatz (VMC Clásico)
    current_params_psi, opt_state_psi, e_mean, e_var, current_cache = trainer.step_ansatz(
        current_walkers, current_params_psi, opt_state_psi
    )
    
    # B) Paso de Optimización del Actor (REINFORCE)
    current_params_actor, opt_state_actor, current_walkers, stats, loss_act, current_cache = trainer.step_actor(
        k_step2, current_walkers, current_params_psi, current_params_actor, opt_state_actor, current_cache
    )
    
    print(f"Paso {step+1}/3 | E_Media: {e_mean:.2f} Ha | Acc: {stats['acceptance']:.2f} | Recompensa: {stats['mean_reward']:.4f}")

print("\n¡Éxito total! Toda la arquitectura compila y los tensores fluyen de manera 100% funcional y JAX-safe.")

Iniciando Ensamblaje del Ecosistema RL-VMC...

--- Compilando y ejecutando pasos de entrenamiento ---

--- Compilando y ejecutando pasos de entrenamiento ---
Paso 1/3 | E_Media: -16.66 Ha | Acc: 0.00 | Recompensa: -0.1000
Paso 2/3 | E_Media: -16.66 Ha | Acc: 0.00 | Recompensa: -0.1000
Paso 3/3 | E_Media: -16.66 Ha | Acc: 0.00 | Recompensa: -0.1000

¡Éxito total! Toda la arquitectura compila y los tensores fluyen de manera 100% funcional y JAX-safe.


In [ ]:
import optax
import jax
import jax.numpy as jnp
from system import PhysicalSystem
from hamiltonians import FullCoulombHamiltonian
from samplers import RLMALASampler
from trainers import GroundStateTrainer

# =====================================================================
# 1. DEFINICIÓN DEL SISTEMA Y LA FÍSICA (LiH)
# =====================================================================
print("1. Inicializando el sistema físico para LiH...")
lih_molecule = PhysicalSystem(
    name="LiH_Singlete",
    n_elec=4,            # 3 electrones del Litio + 1 del Hidrógeno
    n_up=2, n_dn=2,      # Estado basal singlete (espín cero)
    coords=jnp.array([[0.0, 0.0, 0.0], [3.015, 0.0, 0.0]]), # Enlace en unidades atómicas (Bohr)
    charges=jnp.array([3, 1], dtype=jnp.int32)
)

# Acoplamos el Hamiltoniano molecular exacto de Coulomb
hamiltonian = FullCoulombHamiltonian(lih_molecule)

# =====================================================================
# 2. CONFIGURACIÓN DE REDES DE JUGUETE Y OPTIMIZADORES
# =====================================================================
# Redefinimos las funciones dummy con la API corregida de 2 argumentos (sin static_state)
def dummy_ansatz(params, x):
    alpha = params.get("alpha", 0.5)
    return jnp.sum(-alpha * x**2, axis=(-1, -2))

def dummy_actor(params_actor, walkers, elec_idx, nuclei_coords, exploration_mode):
    r_i = walkers[:, elec_idx, :]
    explore_factor = jnp.where(exploration_mode, 1.0, 0.0)
    drift = -params_actor["push"] * r_i * explore_factor
    sigma_raw = jnp.zeros_like(r_i)
    return drift, sigma_raw

# Inicializamos los diccionarios de parámetros ficticios
params_master = {
    "ansatz": {"alpha": 0.45},
    "actor": {"push": 0.01}
}

# Declaramos la estrategia de optimización con Optax
opt_ansatz = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(5e-4))
opt_actor = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(1e-3))

# Inicializamos los estados internos (momentum, varianza) de Adam
opt_state_psi = opt_ansatz.init(params_master["ansatz"])
opt_state_actor = opt_actor.init(params_master["actor"])

# =====================================================================
# 3. ENSAMBLAJE DEL MUESTREADOR Y EL ENTRENADOR
# =====================================================================
# Instanciamos el muestreador aumentado por RL pasando las constantes gamma y beta
rl_sampler = RLMALASampler(
    ansatz_apply=dummy_ansatz,
    actor_apply=dummy_actor,
    batch_size=256,       # Lote de caminantes independientes corriendo en paralelo
    tau=0.005,
    gamma=0.01,
    beta=10.0,
    system=lih_molecule
)

# Construimos el entrenador con las ventanas de clipping seguras para LiH
trainer = GroundStateTrainer(
    system=lih_molecule,
    hamiltonian=hamiltonian,
    sampler=rl_sampler,
    opt_ansatz=opt_ansatz,
    opt_actor=opt_actor,
    energy_clip_min=-50.0, # Techo de seguridad molecular
    energy_clip_max=0.0
)

# =====================================================================
# 4. CONDICIONES INICIALES DEL ESPACIO DE CONFIGURACIÓN
# =====================================================================
key = jax.random.PRNGKey(123)
key, k_walk = jax.random.split(key)

# Generamos los caminantes con las dimensiones exactas del sistema (4 electrones)
walkers = jax.random.normal(k_walk, (rl_sampler.batch_size, lih_molecule.n_elec, 3)) * 0.8

# Establecemos el caché inicial del estado 0 evaluando la red analítica
init_log_psi = dummy_ansatz(params_master["ansatz"], walkers)
init_grad = jax.grad(lambda w: jnp.sum(dummy_ansatz(params_master["ansatz"], w)))(walkers)

cache = {
    "log_psi_curr": init_log_psi,
    "grad_curr": init_grad
}

# =====================================================================
# 5. BUCLE DE ENTRENAMIENTO PRINCIPAL (RUNNER LIGERO)
# =====================================================================
print("\n2. Iniciando corrida ligera de demostración (10 Pasos)...")
print(f"{'Paso':<6} | {'Energía Media':<15} | {'Varianza E':<12} | {'Aceptación':<10} | {'Recompensa':<10} | {'EJD':<8}")
print("-" * 75)

for step in range(1, 11):
    key, k_ansatz, k_actor = jax.random.split(key, 3)
    
    # A) Optimización Variacional de la Función de Onda (Física)
    params_master["ansatz"], opt_state_psi, e_mean, e_var, cache = trainer.step_ansatz(
        walkers, params_master["ansatz"], opt_state_psi
    )
    
    # B) Optimización de la Política del Muestreador (IA)
    params_master["actor"], opt_state_actor, walkers, stats, loss_act, cache = trainer.step_actor(
        k_actor, walkers, params_master["ansatz"], params_master["actor"], opt_state_actor, cache
    )
    
    # Imprimimos el progreso en tiempo real
    print(f"{step:<6} | {e_mean:<15.4f} | {e_var:<12.4f} | {stats['acceptance']:<10.2f} | {stats['mean_reward']:<10.4f} | {stats['ejd']:.4f}")

print("-" * 75)
print("[EXITO] Ciclo terminado de forma JAX-safe. Listos para escalar a DeepQMCAnsatz.")